# 🛰️ Sentinel Protocol — Autonomous Decision-Time-Budget Engine

**Project:** Sentinel Protocol  
**Purpose:** This notebook implements a real-time threat classification engine for a planetary rover AI operating under communication latency constraints with Earth. Because a round-trip signal to Earth can take anywhere from 4 to 24+ minutes, the rover must autonomously decide whether a detected hazard allows time to wait for ground control guidance, requires a safe holding action while Earth is notified, or demands immediate autonomous action.

## Decision Tiers

| Tier | Label | Meaning |
|------|-------|---------|
| 1 | 🟢 GREEN | Time-to-harm exceeds 2× round-trip comm delay — wait for Earth response |
| 2 | 🟡 YELLOW | Time-to-harm is between 1× and 2× round-trip delay — take safe holding action, notify Earth |
| 3 | 🔴 RED | Time-to-harm is less than or equal to 1× round-trip delay — act immediately, notify Earth after |

---

In [1]:
from dataclasses import dataclass, field
from enum import Enum
from typing import Literal
import textwrap


# ---------------------------------------------------------------------------
# Tier enumeration
# ---------------------------------------------------------------------------

class DecisionTier(Enum):
    GREEN  = "GREEN"   # safe to wait for Earth
    YELLOW = "YELLOW"  # hold + notify Earth
    RED    = "RED"     # act now, notify later


# ---------------------------------------------------------------------------
# Threat dataclass
# ---------------------------------------------------------------------------

ThreatType = Literal[
    "cliff_edge",
    "dust_storm",
    "battery_critical",
    "rockfall",
    "comms_blackout",
]

@dataclass
class Threat:
    """Represents a detected hazard encountered by the rover.

    Attributes
    ----------
    threat_type       : One of the five recognised threat categories.
    time_to_harm_s    : Estimated seconds until the hazard causes irreversible
                        damage or mission loss if no action is taken.
    comm_delay_s      : One-way communication delay to Earth in seconds.
                        A full command round-trip therefore costs 2 × this value.
    """
    threat_type:    ThreatType
    time_to_harm_s: float
    comm_delay_s:   float

    # Derived convenience properties
    @property
    def round_trip_s(self) -> float:
        """Full round-trip comm delay (signal to Earth + command back)."""
        return self.comm_delay_s * 2

    @property
    def time_margin_ratio(self) -> float:
        """Ratio of time-to-harm to round-trip delay.  >2 → GREEN, 1-2 → YELLOW, ≤1 → RED."""
        if self.round_trip_s == 0:
            return float("inf")
        return self.time_to_harm_s / self.round_trip_s


# ---------------------------------------------------------------------------
# Threat-specific urgency multipliers
# ---------------------------------------------------------------------------
# Some threats are inherently more dynamic (fast-moving rockfall) or
# statistically under-estimated (cliff_edge sensor noise).  A multiplier
# < 1 applies a conservatism penalty, effectively shrinking the perceived
# time window and pushing ambiguous cases toward higher tiers.

THREAT_CONSERVATISM: dict[str, float] = {
    "cliff_edge":       0.80,   # sensor noise → be conservative
    "dust_storm":       0.90,   # storm intensity can escalate quickly
    "battery_critical": 0.95,   # discharge rate is fairly predictable
    "rockfall":         0.70,   # highly dynamic, worst-case bias
    "comms_blackout":   1.00,   # predictable orbital geometry
}


# ---------------------------------------------------------------------------
# Core classification function
# ---------------------------------------------------------------------------

def classify_threat(
    threat_type:    ThreatType,
    time_to_harm_s: float,
    comm_delay_s:   float,
) -> DecisionTier:
    """Classify a rover threat into a decision tier.

    Parameters
    ----------
    threat_type       : Category of the detected hazard.
    time_to_harm_s    : Estimated seconds to irreversible harm.
    comm_delay_s      : One-way comm latency to Earth in seconds.

    Returns
    -------
    DecisionTier
        GREEN  — time_to_harm > 2 × round_trip  (adjusted for conservatism)
        YELLOW — round_trip < time_to_harm ≤ 2 × round_trip
        RED    — time_to_harm ≤ round_trip
    """
    if threat_type not in THREAT_CONSERVATISM:
        raise ValueError(f"Unknown threat type: {threat_type!r}")

    conservatism = THREAT_CONSERVATISM[threat_type]
    threat = Threat(
        threat_type=threat_type,
        time_to_harm_s=time_to_harm_s * conservatism,   # adjusted window
        comm_delay_s=comm_delay_s,
    )

    ratio = threat.time_margin_ratio

    if ratio > 2.0:
        return DecisionTier.GREEN
    elif ratio > 1.0:
        return DecisionTier.YELLOW
    else:
        return DecisionTier.RED


print("✅ Threat dataclass, classify_threat(), and conservatism table loaded.")

✅ Threat dataclass, classify_threat(), and conservatism table loaded.


---
## Test Cases — All Five Threat Types

Values are chosen to reflect realistic Mars-mission scenarios.  
Mars one-way comm delay ≈ 4–24 min (240–1440 s); used 780 s (~13 min) as a representative mid-range value.

In [2]:
# ---------------------------------------------------------------------------
# Test suite
# ---------------------------------------------------------------------------

COMM_DELAY_S = 780   # ~13-minute one-way delay (realistic Mars mid-range)

test_cases = [
    # (label, threat_type, time_to_harm_s, comm_delay_s, expected_tier)
    (
        "Cliff edge — rover is 4 m from a precipice, moving at 0.02 m/s\n"
        "  → time to reach edge ≈ 200 s.  Far less than round-trip delay.",
        "cliff_edge",
        200,          # 200 s to reach the edge
        COMM_DELAY_S,
        DecisionTier.RED,
    ),
    (
        "Dust storm — approaching storm front detected 90 min away\n"
        "  → 5400 s before solar panels are critically obscured.",
        "dust_storm",
        5400,         # 90 minutes
        COMM_DELAY_S,
        DecisionTier.GREEN,
    ),
    (
        "Battery critical — charge at 8%, estimated 40 min to full shutdown\n"
        "  → 2400 s window; just barely within one round-trip.",
        "battery_critical",
        2400,         # 40 minutes
        COMM_DELAY_S,
        DecisionTier.YELLOW,
    ),
    (
        "Rockfall — seismic sensor detects imminent slope collapse 8 s away\n"
        "  → near-instant hazard, immediate evasion required.",
        "rockfall",
        8,            # 8 seconds — geological event
        COMM_DELAY_S,
        DecisionTier.RED,
    ),
    (
        "Comms blackout — relay satellite occultation in 35 min\n"
        "  → 2100 s before loss of uplink; plenty of time to queue messages.",
        "comms_blackout",
        2100,         # 35 minutes
        COMM_DELAY_S,
        DecisionTier.YELLOW,
    ),
]


# ---------------------------------------------------------------------------
# Run and display
# ---------------------------------------------------------------------------

TIER_ICON = {
    DecisionTier.GREEN:  "🟢",
    DecisionTier.YELLOW: "🟡",
    DecisionTier.RED:    "🔴",
}

TIER_ACTION = {
    DecisionTier.GREEN:  "Wait for Earth response.",
    DecisionTier.YELLOW: "Execute safe holding action; notify Earth immediately.",
    DecisionTier.RED:    "Act autonomously NOW; notify Earth after action.",
}

all_pass = True
print("=" * 70)
print(f"  SENTINEL PROTOCOL — Decision Engine Test Run")
print(f"  Comm delay (one-way): {COMM_DELAY_S} s  |  Round-trip: {COMM_DELAY_S*2} s")
print("=" * 70)

for i, (description, threat_type, tth, comm, expected) in enumerate(test_cases, 1):
    result  = classify_threat(threat_type, tth, comm)
    passed  = result == expected
    all_pass = all_pass and passed
    status  = "PASS ✓" if passed else f"FAIL ✗  (expected {expected.value})"

    # Compute ratio for display (pre-conservatism raw ratio)
    raw_ratio = tth / (comm * 2)
    adj_ratio = (tth * THREAT_CONSERVATISM[threat_type]) / (comm * 2)

    print(f"\nTest {i} — {threat_type.upper().replace('_', ' ')}")
    for line in description.strip().splitlines():
        print(f"  {line}")
    print(f"  Time-to-harm : {tth:,} s")
    print(f"  Raw ratio    : {raw_ratio:.3f}  (time_to_harm / round_trip)")
    print(f"  Adj ratio    : {adj_ratio:.3f}  (after {THREAT_CONSERVATISM[threat_type]:.0%} conservatism)")
    print(f"  Decision     : {TIER_ICON[result]} {result.value}  →  {TIER_ACTION[result]}")
    print(f"  [{status}]")

print("\n" + "=" * 70)
if all_pass:
    print("  ✅ ALL TESTS PASSED — Decision engine nominal.")
else:
    print("  ❌ ONE OR MORE TESTS FAILED — Review classification logic.")
print("=" * 70)

  SENTINEL PROTOCOL — Decision Engine Test Run
  Comm delay (one-way): 780 s  |  Round-trip: 1560 s

Test 1 — CLIFF EDGE
  Cliff edge — rover is 4 m from a precipice, moving at 0.02 m/s
    → time to reach edge ≈ 200 s.  Far less than round-trip delay.
  Time-to-harm : 200 s
  Raw ratio    : 0.128  (time_to_harm / round_trip)
  Adj ratio    : 0.103  (after 80% conservatism)
  Decision     : 🔴 RED  →  Act autonomously NOW; notify Earth after action.
  [PASS ✓]

Test 2 — DUST STORM
  Dust storm — approaching storm front detected 90 min away
    → 5400 s before solar panels are critically obscured.
  Time-to-harm : 5,400 s
  Raw ratio    : 3.462  (time_to_harm / round_trip)
  Adj ratio    : 3.115  (after 90% conservatism)
  Decision     : 🟢 GREEN  →  Wait for Earth response.
  [PASS ✓]

Test 3 — BATTERY CRITICAL
  Battery critical — charge at 8%, estimated 40 min to full shutdown
    → 2400 s window; just barely within one round-trip.
  Time-to-harm : 2,400 s
  Raw ratio    : 1.538  (time

---
## Scenario Simulator — Tick-by-Tick Sensor Escalation

Each threat type has a **physics model** that drives one or more sensor readings forward each tick.  
The raw sensor state is converted to a `time_to_harm_s` estimate, which is fed directly into  
`classify_threat()` so the decision tier updates live as the scenario progresses.

| Threat | Sensor(s) modelled | Escalation mechanism |
|---|---|---|
| `cliff_edge` | `distance_m` (LiDAR) | Rover drifts toward edge each tick |
| `dust_storm` | `wind_speed_ms`, `dust_density_gcm3` | Both ramp up; combined opacity drives shutdown ETA |
| `battery_critical` | `charge_pct` | Discharge accelerates as systems draw more power |
| `rockfall` | `seismic_g`, `debris_distance_m` | Ground vibration rises; debris closes in fast |
| `comms_blackout` | `relay_elevation_deg` | Satellite arc descends toward horizon |

### `run_scenario(threat_type, ticks=20, comm_delay_s=780)`
A generator that yields a `TickState` named-tuple each tick containing:
- `tick` — tick index (0-based)
- `sensors` — `dict` of raw sensor readings for this threat type
- `time_to_harm_s` — derived estimate fed into the classifier
- `tier` — the `DecisionTier` result at this tick

In [3]:
from typing import Iterator, NamedTuple
import math


# ---------------------------------------------------------------------------
# Tick state — what the generator yields each tick
# ---------------------------------------------------------------------------

class TickState(NamedTuple):
    tick:           int
    sensors:        dict        # raw physics readings
    time_to_harm_s: float       # derived estimate → classify_threat input
    tier:           DecisionTier
    holding_action: str | None  # 'hold_in_place' | 'reposition_to_safety' | None


# ---------------------------------------------------------------------------
# Per-threat physics models
# Each model is a generator that yields (sensors_dict, time_to_harm_s) per tick.
# ---------------------------------------------------------------------------

def _cliff_edge_model(ticks: int):
    """Rover drifts toward a precipice; wheel-slip accelerates closure rate."""
    distance_m   = 100.0   # initial LiDAR reading to cliff edge (metres)
    speed_ms     = 0.02    # initial drift speed m/s
    accel        = 0.003   # speed increase per tick (slip worsens)
    tick_dur_s   = 30      # real seconds represented by one tick
    for _ in range(ticks):
        time_to_harm = distance_m / speed_ms if speed_ms > 0 else float('inf')
        yield ({'distance_m': round(distance_m, 3),
                'drift_speed_ms': round(speed_ms, 4)},
               time_to_harm)
        distance_m = max(0.0, distance_m - speed_ms * tick_dur_s)
        speed_ms  += accel


def _dust_storm_model(ticks: int):
    """Wind and dust density rise; combined optical depth predicts panel shutdown."""
    wind_ms      = 4.0     # m/s — gentle breeze
    dust_gcm3    = 0.001   # g/cm³ — background haze
    wind_ramp    = 2.5     # m/s added per tick
    dust_ramp    = 0.004   # density increase per tick
    # Shutdown threshold: panels fail when optical_depth > 1.0
    # optical_depth proxy = dust_gcm3 * 100 * wind_ms^0.3
    # time_to_harm = seconds until optical_depth reaches 1.0, linear extrapolation
    tick_dur_s   = 60
    for _ in range(ticks):
        optical_depth = dust_gcm3 * 100 * (wind_ms ** 0.3)
        # rate of optical_depth change per second
        next_dust  = dust_gcm3 + dust_ramp
        next_wind  = wind_ms  + wind_ramp
        next_od    = next_dust * 100 * (next_wind ** 0.3)
        od_rate_per_s = max((next_od - optical_depth) / tick_dur_s, 1e-9)
        remaining_od  = max(1.0 - optical_depth, 0.0)
        time_to_harm  = remaining_od / od_rate_per_s
        yield ({'wind_speed_ms':     round(wind_ms, 2),
                'dust_density_gcm3': round(dust_gcm3, 4),
                'optical_depth':     round(optical_depth, 4)},
               max(time_to_harm, 0.0))
        wind_ms   = next_wind
        dust_gcm3 = next_dust


def _battery_critical_model(ticks: int):
    """Battery charge drains; draw rate accelerates as thermal systems kick in."""
    charge_pct   = 18.0    # %
    draw_pct_per_tick = 0.8  # base drain per tick
    draw_accel   = 0.12    # drain increase per tick (heaters, comms fighting)
    tick_dur_s   = 60
    for _ in range(ticks):
        # time to reach 0 % at current draw rate
        time_to_harm = (charge_pct / draw_pct_per_tick) * tick_dur_s
        yield ({'charge_pct':      round(charge_pct, 2),
                'draw_pct_per_tick': round(draw_pct_per_tick, 3)},
               time_to_harm)
        charge_pct      = max(0.0, charge_pct - draw_pct_per_tick)
        draw_pct_per_tick += draw_accel


def _rockfall_model(ticks: int):
    """Seismic intensity rises; debris front closes rapidly."""
    seismic_g     = 0.05   # gravitational acceleration units
    debris_dist_m = 80.0   # metres to the debris front
    debris_speed  = 1.5    # m/s — slow initial roll
    seismic_ramp  = 0.08
    speed_ramp    = 2.0    # debris accelerates under gravity
    tick_dur_s    = 5
    for _ in range(ticks):
        time_to_harm = debris_dist_m / debris_speed if debris_speed > 0 else float('inf')
        yield ({'seismic_g':      round(seismic_g, 3),
                'debris_dist_m':  round(debris_dist_m, 2),
                'debris_speed_ms': round(debris_speed, 2)},
               time_to_harm)
        debris_dist_m = max(0.0, debris_dist_m - debris_speed * tick_dur_s)
        debris_speed += speed_ramp
        seismic_g    += seismic_ramp


def _comms_blackout_model(ticks: int):
    """Relay satellite arc descends; time to LOS shrinks linearly then faster near horizon."""
    elevation_deg = 42.0   # degrees above horizon
    descent_rate  = 1.8    # deg per tick
    tick_dur_s    = 60
    for _ in range(ticks):
        # Time to reach 5-deg minimum elevation (operational cutoff)
        remaining_deg = max(elevation_deg - 5.0, 0.0)
        # descent accelerates slightly due to orbital geometry near horizon
        effective_rate = descent_rate * (1 + 0.04 * (42.0 - elevation_deg))
        time_to_harm   = (remaining_deg / effective_rate) * tick_dur_s
        yield ({'relay_elevation_deg': round(elevation_deg, 2),
                'effective_descent_rate': round(effective_rate, 3)},
               max(time_to_harm, 0.0))
        elevation_deg = max(0.0, elevation_deg - descent_rate)


_MODELS = {
    'cliff_edge':       _cliff_edge_model,
    'dust_storm':       _dust_storm_model,
    'battery_critical': _battery_critical_model,
    'rockfall':         _rockfall_model,
    'comms_blackout':   _comms_blackout_model,
}


# ---------------------------------------------------------------------------
# YELLOW-tier holding action selector
# (defined here so run_scenario can call it; also re-exposed in its own section)
# ---------------------------------------------------------------------------

_REPOSITION_UNSAFE_WIND_MS = 20.0   # m/s — too dangerous to traverse
_REPOSITION_UNSAFE_CHARGE  =  5.0   # %   — insufficient power to reposition

def choose_holding_action(
    threat_type:  str,
    sensor_state: dict,
    comm_delay_s: float = 780,
) -> str:
    """Return 'hold_in_place' or 'reposition_to_safety' for a YELLOW-tier tick.

    Rules
    -----
    cliff_edge      -> hold_in_place         (any movement near a cliff is unsafe)
    dust_storm      -> reposition_to_safety  (unless wind >= 20 m/s)
    battery_critical-> reposition_to_safety  (unless charge <= 5 %)
    rockfall        -> hold_in_place         (movement increases debris exposure)
    comms_blackout  -> hold_in_place         (navigating blind is unsafe)
    """
    if threat_type == 'cliff_edge':
        return 'hold_in_place'
    if threat_type == 'dust_storm':
        return ('hold_in_place'
                if sensor_state.get('wind_speed_ms', 0.0) >= _REPOSITION_UNSAFE_WIND_MS
                else 'reposition_to_safety')
    if threat_type == 'battery_critical':
        return ('hold_in_place'
                if sensor_state.get('charge_pct', 100.0) <= _REPOSITION_UNSAFE_CHARGE
                else 'reposition_to_safety')
    if threat_type in ('rockfall', 'comms_blackout'):
        return 'hold_in_place'
    return 'hold_in_place'


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def run_scenario(
    threat_type:  ThreatType,
    ticks:        int   = 20,
    comm_delay_s: float = 780,
) -> Iterator[TickState]:
    """Tick-by-tick scenario simulator for a single threat type.

    Yields
    ------
    TickState
        tick           : Tick index (0-based).
        sensors        : Raw sensor readings dict for the current tick.
        time_to_harm_s : Derived time-to-harm estimate (seconds).
        tier           : DecisionTier result from classify_threat().
        holding_action : 'hold_in_place' | 'reposition_to_safety' | None.
                         Non-None only on YELLOW ticks; None on GREEN and RED.
    """
    if threat_type not in _MODELS:
        raise ValueError(f'Unknown threat type: {threat_type!r}')
    model = _MODELS[threat_type](ticks)
    for tick_idx, (sensors, tth) in enumerate(model):
        tier = classify_threat(threat_type, tth, comm_delay_s)
        ha   = (choose_holding_action(threat_type, sensors, comm_delay_s)
                if tier == DecisionTier.YELLOW else None)
        yield TickState(tick=tick_idx, sensors=sensors,
                        time_to_harm_s=round(tth, 1), tier=tier,
                        holding_action=ha)


print('Scenario simulator loaded. Models: ' + ', '.join(_MODELS))

Scenario simulator loaded. Models: cliff_edge, dust_storm, battery_critical, rockfall, comms_blackout


---
### Demo Run — `cliff_edge` scenario (20 ticks)

The rover starts **12 m** from the edge drifting at **0.02 m/s**.  
Wheel-slip gradually increases the closure rate each tick until the decision escalates  
from 🟢 GREEN → 🟡 YELLOW → 🔴 RED.

In [4]:
TIER_ICON = {
    DecisionTier.GREEN:  'GREEN ',
    DecisionTier.YELLOW: 'YELLOW',
    DecisionTier.RED:    'RED   ',
}
TIER_BADGE = {
    DecisionTier.GREEN:  '[GREEN ] -->  Wait for Earth response.',
    DecisionTier.YELLOW: '[YELLOW] -->  Safe hold + notify Earth immediately.',
    DecisionTier.RED:    '[RED   ] -->  ACT NOW — notify Earth after.',
}

COMM_DELAY_S = 780

print('=' * 72)
print('  SENTINEL PROTOCOL — Scenario Simulator')
print('  Scenario : cliff_edge')
print(f'  Comm delay (one-way): {COMM_DELAY_S} s  |  Round-trip: {COMM_DELAY_S*2} s')
print('  Tick duration: 30 s real-world seconds per tick')
print('=' * 72)
print(f'  {"Tick":>4}  {"Dist(m)":>8}  {"Speed(m/s)":>10}  {"TTH(s)":>9}  {"Tier & Action"}')
print('-' * 72)

prev_tier = None
for state in run_scenario('cliff_edge', ticks=20, comm_delay_s=COMM_DELAY_S):
    d   = state.sensors['distance_m']
    spd = state.sensors['drift_speed_ms']
    tth = state.time_to_harm_s
    tier = state.tier

    # Mark tier transitions
    transition = '  <-- TIER CHANGE' if tier != prev_tier and prev_tier is not None else ''
    prev_tier = tier

    # Append holding action label on YELLOW ticks
    ha_label = ''
    if state.holding_action is not None:
        ha_label = f'  [{state.holding_action}]'

    print(f'  {state.tick:>4}  {d:>8.3f}  {spd:>10.4f}  {tth:>9.1f}  {TIER_BADGE[tier]}{ha_label}{transition}')

print('=' * 72)
print('  Simulation complete.')
print('=' * 72)

  SENTINEL PROTOCOL — Scenario Simulator
  Scenario : cliff_edge
  Comm delay (one-way): 780 s  |  Round-trip: 1560 s
  Tick duration: 30 s real-world seconds per tick
  Tick   Dist(m)  Speed(m/s)     TTH(s)  Tier & Action
------------------------------------------------------------------------
     0   100.000      0.0200     5000.0  [GREEN ] -->  Wait for Earth response.
     1    99.400      0.0230     4321.7  [GREEN ] -->  Wait for Earth response.
     2    98.710      0.0260     3796.5  [YELLOW] -->  Safe hold + notify Earth immediately.  [hold_in_place]  <-- TIER CHANGE
     3    97.930      0.0290     3376.9  [YELLOW] -->  Safe hold + notify Earth immediately.  [hold_in_place]
     4    97.060      0.0320     3033.1  [YELLOW] -->  Safe hold + notify Earth immediately.  [hold_in_place]
     5    96.100      0.0350     2745.7  [YELLOW] -->  Safe hold + notify Earth immediately.  [hold_in_place]
     6    95.050      0.0380     2501.3  [YELLOW] -->  Safe hold + notify Earth immedia

---
## AI Reasoning Layer — IBM watsonx.ai (Granite)

Each tick's structured data is sent to a **watsonx.ai foundation model** (`ibm/granite-4-h-small`,
Frankfurt endpoint) which generates a single professional mission-log sentence — the kind a flight
engineer would write to justify an autonomous decision.

### Credentials
Loaded from a local `.env` file (never committed to git):
```
WATSONX_API_KEY=<your IBM Cloud API key>
WATSONX_PROJECT_ID=<your watsonx.ai project GUID>
```

### `generate_reasoning(tick_data)` — input schema
```python
{
    'threat_type':    str,   # e.g. 'cliff_edge'
    'sensors':        dict,  # raw sensor readings at this tick
    'time_to_harm_s': float, # seconds to irreversible harm
    'round_trip_s':   float, # full comm round-trip delay in seconds
    'ratio':          float, # adjusted TTH / round-trip ratio
    'tier':           str,   # 'GREEN' | 'YELLOW' | 'RED'
    'action':         str,   # prescribed action string
}
```

In [5]:
import os
from dotenv import load_dotenv
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference

# ---------------------------------------------------------------------------
# Load credentials from .env
# ---------------------------------------------------------------------------
load_dotenv()  # reads .env from the current working directory

WATSONX_API_KEY    = os.environ['WATSONX_API_KEY']
WATSONX_PROJECT_ID = os.environ['WATSONX_PROJECT_ID']
WATSONX_URL        = 'https://eu-de.ml.cloud.ibm.com'
WATSONX_MODEL_ID   = 'ibm/granite-4-h-small'

# ---------------------------------------------------------------------------
# Instantiate the model client
# granite-4-h-small is a chat model — use the /ml/v1/text/chat endpoint
# via .chat(), NOT .generate_text() which targets the deprecated completion API.
# ---------------------------------------------------------------------------
_wx_model = ModelInference(
    model_id    = WATSONX_MODEL_ID,
    credentials = Credentials(
        url     = WATSONX_URL,
        api_key = WATSONX_API_KEY,
    ),
    project_id  = WATSONX_PROJECT_ID,
)

# Chat parameters passed per-call (not stored on the client for chat API)
_CHAT_PARAMS = {
    'max_tokens':        80,
    'temperature':       0.3,
    'repetition_penalty': 1.05,
}


# ---------------------------------------------------------------------------
# System prompt (sets the persona once, reused across calls)
# ---------------------------------------------------------------------------
_SYSTEM_PROMPT = (
    'You are the autonomous reasoning system of a planetary rover named Sentinel. '
    'Write a single professional mission-log sentence (maximum 40 words) explaining '
    'the decision made. Be factual, precise, and terse — like a flight engineer '
    'writing a flight log entry. Output only the log sentence, nothing else.'
)


# ---------------------------------------------------------------------------
# User message template (the structured situation fed per tick)
# ---------------------------------------------------------------------------
_USER_TEMPLATE = (
    'SITUATION:\n'
    '  Threat      : {threat_type}\n'
    '  Sensor data : {sensors}\n'
    '  Time-to-harm: {time_to_harm_s:.1f} s\n'
    '  Round-trip comm delay: {round_trip_s:.0f} s\n'
    '  Adjusted ratio (TTH/RTT): {ratio:.3f}\n'
    '  Decision tier: {tier}\n'
    '  Required action: {action}\n'
    '\n'
    'Write the mission log entry for this tick.'
)


# ---------------------------------------------------------------------------
# Public function
# ---------------------------------------------------------------------------

def generate_reasoning(tick_data: dict) -> str:
    """Call watsonx.ai to generate a mission-log sentence for one tick.

    Uses the chat API (/ml/v1/text/chat) which is the correct endpoint for
    ibm/granite-4-h-small and avoids the deprecated text/generation endpoint.

    Parameters
    ----------
    tick_data : dict with keys:
        threat_type, sensors, time_to_harm_s, round_trip_s, ratio, tier, action

    Returns
    -------
    str  — a single mission-log sentence generated by the foundation model.
    """
    messages = [
        {'role': 'system', 'content': _SYSTEM_PROMPT},
        {'role': 'user',   'content': _USER_TEMPLATE.format(**tick_data)},
    ]
    response = _wx_model.chat(messages=messages, params=_CHAT_PARAMS)
    return response['choices'][0]['message']['content'].strip()


print(f'watsonx chat client ready  model={WATSONX_MODEL_ID}  url={WATSONX_URL}')

watsonx chat client ready  model=ibm/granite-4-h-small  url=https://eu-de.ml.cloud.ibm.com


---
### Live Demo — `generate_reasoning()` at tick 9 (GREEN → RED transition)

Tick 9 is the first **RED** tick in the `cliff_edge` simulation: distance has closed to
**91.36 m**, drift speed has accelerated to **0.047 m/s**, time-to-harm is **1943.8 s** —
below the 1560 s round-trip threshold. The AI layer explains *why* the rover must act now.

In [6]:
# ---------------------------------------------------------------------------
# Reconstruct tick-9 state from the cliff_edge scenario
# ---------------------------------------------------------------------------
COMM_DELAY_S = 780

tick9_state = None
for state in run_scenario('cliff_edge', ticks=20, comm_delay_s=COMM_DELAY_S):
    if state.tick == 9:
        tick9_state = state
        break

# Build the tick_data dict that generate_reasoning() expects
conservatism  = THREAT_CONSERVATISM['cliff_edge']
round_trip_s  = COMM_DELAY_S * 2
adj_tth       = tick9_state.time_to_harm_s * conservatism
ratio         = adj_tth / round_trip_s

TIER_ACTION_MAP = {
    DecisionTier.GREEN:  'Wait for Earth response.',
    DecisionTier.YELLOW: 'Execute safe holding action; notify Earth immediately.',
    DecisionTier.RED:    'Act autonomously NOW; notify Earth after action.',
}

tick_data = {
    'threat_type':    'cliff_edge',
    'sensors':        tick9_state.sensors,
    'time_to_harm_s': tick9_state.time_to_harm_s,
    'round_trip_s':   round_trip_s,
    'ratio':          ratio,
    'tier':           tick9_state.tier.value,
    'action':         TIER_ACTION_MAP[tick9_state.tier],
}

# ---------------------------------------------------------------------------
# Print tick summary
# ---------------------------------------------------------------------------
print('=' * 68)
print('  SENTINEL PROTOCOL — AI Reasoning Layer Demo')
print('  Tick 9 | cliff_edge | TIER TRANSITION: YELLOW -> RED')
print('=' * 68)
print(f"  Threat      : {tick_data['threat_type']}")
print(f"  Sensors     : {tick_data['sensors']}")
print(f"  Time-to-harm: {tick_data['time_to_harm_s']} s")
print(f"  Round-trip  : {tick_data['round_trip_s']} s")
print(f"  Adj ratio   : {tick_data['ratio']:.3f}")
print(f"  Tier        : {tick_data['tier']}")
print(f"  Action      : {tick_data['action']}")
print('-' * 68)
print('  Querying watsonx.ai...')

# ---------------------------------------------------------------------------
# Call the model
# ---------------------------------------------------------------------------
log_entry = generate_reasoning(tick_data)

print()
print('  MISSION LOG ENTRY (generated by ibm/granite-4-h-small):')
print(f'  "{log_entry}"')
print('=' * 68)

  SENTINEL PROTOCOL — AI Reasoning Layer Demo
  Tick 9 | cliff_edge | TIER TRANSITION: YELLOW -> RED
  Threat      : cliff_edge
  Sensors     : {'distance_m': 91.36, 'drift_speed_ms': 0.047}
  Time-to-harm: 1943.8 s
  Round-trip  : 1560 s
  Adj ratio   : 0.997
  Tier        : RED
  Action      : Act autonomously NOW; notify Earth after action.
--------------------------------------------------------------------
  Querying watsonx.ai...



  MISSION LOG ENTRY (generated by ibm/granite-4-h-small):
  "Autonomously executed evasive maneuver to avoid cliff edge; will transmit detailed report post-action completion."


---
## Command Validation Layer

Ground control sends commands to the rover across a comm link that may be delayed by 4–24 minutes.
By the time a command arrives, the rover's situation may have changed significantly.  
The **command validation layer** intercepts every incoming command and checks it against the
rover's current sensor state *before* execution.  If executing the command would worsen or
ignore an active hazard, it is **BLOCKED** and the rover holds position while reporting back
to Earth with a watsonx-generated plain-language explanation.

### `validate_command(command, sensor_state, threat_type, comm_delay_s)` — signature

| Parameter | Type | Description |
|---|---|---|
| `command` | `str` | Incoming command string from Earth |
| `sensor_state` | `dict` | Current sensor readings (same structure as simulator tick) |
| `threat_type` | `str` | Active threat category (or `None` if no active threat) |
| `comm_delay_s` | `float` | One-way comm delay in seconds |

### Returns: `ValidationResult` namedtuple

| Field | Type | Values |
|---|---|---|
| `verdict` | `str` | `'APPROVED'` or `'BLOCKED'` |
| `command` | `str` | The original command (passed through unchanged if approved) |
| `reason` | `str` | Human-readable conflict description (empty string if approved) |
| `earth_report` | `str` | watsonx-generated message back to Earth (empty string if approved) |

### Conflict rules by threat type

| Threat | Blocked command group | Block condition |
|---|---|---|
| `cliff_edge` | advance commands | adj. time-to-edge ≤ round-trip delay (1× RTT) |
| `dust_storm` | antenna / high-drag commands | wind ≥ 15 m/s **or** optical depth ≥ 0.6 |
| `battery_critical` | high-power commands | charge ≤ 10 % |
| `rockfall` | any movement | debris ETA ≤ 30 s |
| `comms_blackout` | transmission commands | relay elevation ≤ 8 ° |

Retreat / stop / hold commands are **never blocked** — the rover can always move away from danger.

In [7]:
from typing import NamedTuple


# ---------------------------------------------------------------------------
# Command groups
# ---------------------------------------------------------------------------

# Commands that move the rover in the forward / current direction
_ADVANCE_CMDS = frozenset({
    'move_forward', 'continue_heading', 'increase_speed',
    'resume_traverse', 'proceed', 'advance',
})

# Commands that deploy or extend structural elements in high-wind/dust
_ANTENNA_CMDS = frozenset({
    'deploy_antenna', 'raise_antenna', 'extend_mast',
    'open_solar_panel', 'deploy_instrument',
})

# Commands that draw significant electrical power
_HIGH_POWER_CMDS = frozenset({
    'deploy_antenna', 'raise_antenna', 'transmit_data',
    'queue_transmission', 'activate_drill', 'run_diagnostics',
    'enable_heaters', 'full_sensor_sweep',
})

# Any command that causes physical movement
_MOVEMENT_CMDS = frozenset({
    'move_forward', 'continue_heading', 'increase_speed',
    'resume_traverse', 'proceed', 'advance',
    'move_backward', 'reverse', 'turn_left', 'turn_right',
    'change_heading', 'reposition',
})

# Commands that attempt to transmit or queue data
_COMMS_CMDS = frozenset({
    'transmit_data', 'queue_transmission', 'send_telemetry',
    'uplink_report', 'broadcast_status',
})


# ---------------------------------------------------------------------------
# Result type
# ---------------------------------------------------------------------------

class ValidationResult(NamedTuple):
    verdict:      str   # 'APPROVED' or 'BLOCKED'
    command:      str   # original command, unchanged
    reason:       str   # conflict description (empty if approved)
    earth_report: str   # watsonx-generated message back to Earth


# ---------------------------------------------------------------------------
# Watsonx prompt for the Earth-report (separate from the mission-log prompt)
# ---------------------------------------------------------------------------

_BLOCK_REPORT_SYSTEM = (
    'You are the autonomous safety system of a planetary rover named Sentinel. '
    'A command from Earth has been blocked because it would create a hazard conflict. '
    'Write a single professional sentence (maximum 45 words) reporting the block back to Earth. '
    'Be factual, concise, and terse — the way a flight engineer would write a status update. '
    'Output only the sentence, nothing else.'
)

_BLOCK_REPORT_USER = (
    'Blocked command : {command}\n'
    'Active threat   : {threat_type}\n'
    'Sensor state    : {sensors}\n'
    'Conflict reason : {reason}\n'
    '\n'
    'Write the status report back to Earth.'
)


def _generate_block_report(command: str, threat_type: str,
                            sensors: dict, reason: str) -> str:
    """Call watsonx.ai to generate a plain-language Earth-facing block report."""
    # Re-use the already-initialised _wx_model from the watsonx setup cell.
    # Falls back gracefully if credentials are not configured.
    try:
        messages = [
            {'role': 'system', 'content': _BLOCK_REPORT_SYSTEM},
            {'role': 'user',   'content': _BLOCK_REPORT_USER.format(
                command=command,
                threat_type=threat_type,
                sensors=str(sensors),
                reason=reason,
            )},
        ]
        resp = _wx_model.chat(
            messages=messages,
            params={'max_tokens': 90, 'temperature': 0.3, 'repetition_penalty': 1.05},
        )
        return resp['choices'][0]['message']['content'].strip()
    except Exception as e:
        return f'(watsonx unavailable: {e})'


# ---------------------------------------------------------------------------
# Conflict detectors — one per threat type
# Each returns (conflict: bool, reason: str)
# ---------------------------------------------------------------------------

def _check_cliff_edge(command: str, sensors: dict, comm_delay_s: float):
    """Block advance commands when adj. time-to-edge <= round-trip delay."""
    if command not in _ADVANCE_CMDS:
        return False, ''
    dist_m  = sensors.get('distance_m', float('inf'))
    speed   = sensors.get('drift_speed_ms', 0.0)
    if speed <= 0:
        return False, ''
    tth_raw = dist_m / speed                          # raw seconds to edge
    tth_adj = tth_raw * THREAT_CONSERVATISM['cliff_edge']  # apply conservatism
    rtt     = comm_delay_s * 2
    if tth_adj <= rtt:
        return True, (
            f'cliff edge {dist_m:.1f} m ahead at current drift rate; '
            f'adjusted time-to-edge {tth_adj:.0f} s ≤ round-trip delay {rtt:.0f} s'
        )
    return False, ''


def _check_dust_storm(command: str, sensors: dict, comm_delay_s: float):
    """Block structural deployments when wind or opacity exceeds safe limits."""
    if command not in _ANTENNA_CMDS:
        return False, ''
    wind  = sensors.get('wind_speed_ms', 0.0)
    opdep = sensors.get('optical_depth', 0.0)
    if wind >= 15.0:
        return True, (
            f'wind speed {wind:.1f} m/s exceeds structural safety limit (15 m/s); '
            f'deployment would risk mast or panel damage'
        )
    if opdep >= 0.6:
        return True, (
            f'dust optical depth {opdep:.3f} exceeds 0.60; '
            f'particulate ingestion risk to deployed mechanisms'
        )
    return False, ''


def _check_battery_critical(command: str, sensors: dict, comm_delay_s: float):
    """Block high-power commands when charge is critically low."""
    if command not in _HIGH_POWER_CMDS:
        return False, ''
    charge = sensors.get('charge_pct', 100.0)
    if charge <= 10.0:
        return True, (
            f'battery at {charge:.1f}% — executing high-power command would '
            f'risk full system shutdown before safe-mode entry'
        )
    return False, ''


def _check_rockfall(command: str, sensors: dict, comm_delay_s: float):
    """Block any movement when debris ETA is under 30 seconds."""
    if command not in _MOVEMENT_CMDS:
        return False, ''
    dist  = sensors.get('debris_dist_m', float('inf'))
    speed = sensors.get('debris_speed_ms', 0.0)
    if speed <= 0:
        return False, ''
    eta = dist / speed
    if eta <= 30.0:
        return True, (
            f'debris front {dist:.1f} m away at {speed:.1f} m/s — '
            f'impact ETA {eta:.1f} s; movement would expose rover to debris'
        )
    return False, ''


def _check_comms_blackout(command: str, sensors: dict, comm_delay_s: float):
    """Block transmission commands when relay satellite is below cutoff elevation."""
    if command not in _COMMS_CMDS:
        return False, ''
    elev = sensors.get('relay_elevation_deg', 90.0)
    if elev <= 8.0:
        return True, (
            f'relay satellite at {elev:.1f}\u00b0 elevation (cutoff 8\u00b0); '
            f'transmission would fail and waste power'
        )
    return False, ''


_CONFLICT_CHECKERS = {
    'cliff_edge':       _check_cliff_edge,
    'dust_storm':       _check_dust_storm,
    'battery_critical': _check_battery_critical,
    'rockfall':         _check_rockfall,
    'comms_blackout':   _check_comms_blackout,
}


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def validate_command(
    command:      str,
    sensor_state: dict,
    threat_type:  str | None = None,
    comm_delay_s: float = 780,
) -> ValidationResult:
    """Validate an incoming Earth command against the rover's current sensor state.

    Parameters
    ----------
    command      : Incoming command string from Earth.
    sensor_state : Current sensor readings dict (same structure as TickState.sensors).
    threat_type  : Active threat category, or None if no threat is active.
    comm_delay_s : One-way comm delay in seconds (default 780 s ≈ 13 min, Mars mid-range).

    Returns
    -------
    ValidationResult
        verdict      : 'APPROVED' or 'BLOCKED'
        command      : Original command string (unchanged).
        reason       : Conflict description (empty string if approved).
        earth_report : watsonx-generated sentence back to Earth (empty if approved).
    """
    cmd = command.strip().lower()

    # No active threat — every command passes
    if threat_type is None or threat_type not in _CONFLICT_CHECKERS:
        return ValidationResult(
            verdict='APPROVED', command=command, reason='', earth_report=''
        )

    checker  = _CONFLICT_CHECKERS[threat_type]
    conflict, reason = checker(cmd, sensor_state, comm_delay_s)

    if not conflict:
        return ValidationResult(
            verdict='APPROVED', command=command, reason='', earth_report=''
        )

    # Generate the Earth-facing block report via watsonx
    earth_report = _generate_block_report(
        command=command,
        threat_type=threat_type,
        sensors=sensor_state,
        reason=reason,
    )

    return ValidationResult(
        verdict='BLOCKED',
        command=command,
        reason=reason,
        earth_report=earth_report,
    )


print('Command validation layer loaded.')
print(f'  Conflict checkers : {list(_CONFLICT_CHECKERS)}')
print(f'  Command groups    : ADVANCE({len(_ADVANCE_CMDS)})  ANTENNA({len(_ANTENNA_CMDS)})  '
      f'HIGH_POWER({len(_HIGH_POWER_CMDS)})  MOVEMENT({len(_MOVEMENT_CMDS)})  '
      f'COMMS({len(_COMMS_CMDS)})')

Command validation layer loaded.
  Conflict checkers : ['cliff_edge', 'dust_storm', 'battery_critical', 'rockfall', 'comms_blackout']
  Command groups    : ADVANCE(6)  ANTENNA(5)  HIGH_POWER(8)  MOVEMENT(12)  COMMS(5)


---
### Test Cases — Command Validation

**Test 1 — BLOCKED:** Ground sends `move_forward` while cliff edge is 91 m ahead  
at 0.047 m/s drift (tick-9 state from the earlier simulation). Adjusted TTH = 1555 s ≤ RTT 1560 s → blocked.

**Test 2 — APPROVED:** Ground sends `hold_position` with the same cliff-edge sensor state.  
Holding position never worsens a cliff hazard → approved.

**Test 3 — APPROVED (no active threat):** Ground sends `move_forward` with an open-terrain  
sensor state and no active threat → approved unconditionally.

In [8]:
COMM_DELAY_S = 780
SEP = '=' * 68
SEP2 = '-' * 68

VERDICT_ICON = {'APPROVED': '[APPROVED]', 'BLOCKED': '[BLOCKED ]'}


def _print_result(n: int, label: str, result: ValidationResult):
    icon = VERDICT_ICON[result.verdict]
    print(SEP)
    print(f'  Test {n} — {label}')
    print(SEP2)
    print(f'  Command  : {result.command}')
    print(f'  Verdict  : {icon} {result.verdict}')
    if result.reason:
        print(f'  Conflict : {result.reason}')
    if result.earth_report:
        print()
        print(f'  Earth report (ibm/granite-4-h-small):')
        print(f'  >>> {result.earth_report}')
    print()


# ---------------------------------------------------------------------------
# Test 1 — move_forward BLOCKED (cliff edge, tick-9 sensor state)
# ---------------------------------------------------------------------------
# Tick 9 from the cliff_edge simulation: distance_m=91.36, drift_speed_ms=0.047
# adj TTH = (91.36 / 0.047) * 0.80 = 1555 s  <=  RTT 1560 s  -> BLOCKED

cliff_sensors_t9 = {'distance_m': 91.36, 'drift_speed_ms': 0.047}

result1 = validate_command(
    command      = 'move_forward',
    sensor_state = cliff_sensors_t9,
    threat_type  = 'cliff_edge',
    comm_delay_s = COMM_DELAY_S,
)
_print_result(1, 'move_forward | cliff_edge active | should be BLOCKED', result1)


# ---------------------------------------------------------------------------
# Test 2 — hold_position APPROVED (same cliff sensor state, safe command)
# ---------------------------------------------------------------------------
# hold_position is not in _ADVANCE_CMDS — rover is staying put, not closing
# the gap. No conflict regardless of cliff proximity.

result2 = validate_command(
    command      = 'hold_position',
    sensor_state = cliff_sensors_t9,
    threat_type  = 'cliff_edge',
    comm_delay_s = COMM_DELAY_S,
)
_print_result(2, 'hold_position | cliff_edge active | should be APPROVED', result2)


# ---------------------------------------------------------------------------
# Test 3 — move_forward APPROVED (open terrain, no active threat)
# ---------------------------------------------------------------------------

open_terrain_sensors = {'distance_m': 850.0, 'drift_speed_ms': 0.02}

result3 = validate_command(
    command      = 'move_forward',
    sensor_state = open_terrain_sensors,
    threat_type  = None,           # no active threat
    comm_delay_s = COMM_DELAY_S,
)
_print_result(3, 'move_forward | no active threat | should be APPROVED', result3)

print(SEP)
all_pass = (
    result1.verdict == 'BLOCKED' and
    result2.verdict == 'APPROVED' and
    result3.verdict == 'APPROVED'
)
if all_pass:
    print('  All 3 validation tests passed.')
else:
    print('  One or more tests failed — review conflict logic.')
print(SEP)

  Test 1 — move_forward | cliff_edge active | should be BLOCKED
--------------------------------------------------------------------
  Command  : move_forward
  Verdict  : [BLOCKED ] BLOCKED
  Conflict : cliff edge 91.4 m ahead at current drift rate; adjusted time-to-edge 1555 s ≤ round-trip delay 1560 s

  Earth report (ibm/granite-4-h-small):
  >>> Command "move_forward" blocked due to detected cliff edge 91.4 meters ahead with current drift speed of 0.047 m/s, resulting in an adjusted time-to-edge of 1555 seconds which is less than or equal to the round-trip communication delay of 1560 seconds.

  Test 2 — hold_position | cliff_edge active | should be APPROVED
--------------------------------------------------------------------
  Command  : hold_position
  Verdict  : [APPROVED] APPROVED

  Test 3 — move_forward | no active threat | should be APPROVED
--------------------------------------------------------------------
  Command  : move_forward
  Verdict  : [APPROVED] APPROVED

  All

---
## YELLOW-Tier Holding Action Selection

When the decision engine returns **YELLOW**, the rover must take a *safe holding action* while
notifying Earth.  Rather than a single generic response, `choose_holding_action()` selects
between two concrete behaviours based on the active threat and current sensor readings:

| Behaviour | Description |
|---|---|
| `hold_in_place` | Stop all movement, maintain current position, preserve power |
| `reposition_to_safety` | Move toward a known safe location (sunlit area / charging dock) |

### Decision rules

| Threat | Default | Override condition |
|---|---|---|
| `cliff_edge` | `hold_in_place` | Never override — moving near a cliff edge is always unsafe |
| `dust_storm` | `reposition_to_safety` | Downgrade to `hold_in_place` if wind ≥ 20 m/s (too dangerous to move) |
| `battery_critical` | `reposition_to_safety` | Downgrade to `hold_in_place` if charge ≤ 5 % (insufficient power to reposition) |
| `rockfall` | `hold_in_place` | Never override — any movement increases debris exposure |
| `comms_blackout` | `hold_in_place` | Never override — moving blind without comms guidance is unsafe |

> `choose_holding_action()` is called automatically by `run_scenario()` on every YELLOW tick,
> so its result appears in `TickState.holding_action` throughout the simulator and dashboard.

In [9]:
# choose_holding_action() is defined in the simulator cell above (cell 5).
# This cell just confirms the function and its thresholds are accessible.
_REPOSITION_UNSAFE_WIND_MS  = 20.0   # m/s — structural risk if moving in storm
_REPOSITION_UNSAFE_CHARGE   =  5.0   # % — not enough power to navigate safely

print('choose_holding_action() ready.')
print(f'  Reposition blocked if: wind >= {_REPOSITION_UNSAFE_WIND_MS} m/s  '
      f'or charge <= {_REPOSITION_UNSAFE_CHARGE} %')

choose_holding_action() ready.
  Reposition blocked if: wind >= 20.0 m/s  or charge <= 5.0 %


---
### Test Cases — YELLOW Holding Action Selection

Each test calls `choose_holding_action()` directly with sensor states chosen to exercise
both outcomes, then runs the corresponding scenario through the full simulator to confirm
the holding action appears correctly on every YELLOW tick in `TickState.holding_action`.

| Test | Threat | Sensor state | Expected action |
|---|---|---|---|
| 1 | `battery_critical` | charge = 12 % (above 5 % floor) | `reposition_to_safety` |
| 2 | `battery_critical` | charge = 3 % (below 5 % floor) | `hold_in_place` |
| 3 | `dust_storm` | wind = 10 m/s (below 20 m/s ceiling) | `reposition_to_safety` |
| 4 | `dust_storm` | wind = 22 m/s (above 20 m/s ceiling) | `hold_in_place` |
| 5 | `cliff_edge` | any | `hold_in_place` |
| 6 | `rockfall` | any | `hold_in_place` |
| 7 | `comms_blackout` | any | `hold_in_place` |
| 8 | Simulator YELLOW ticks | `battery_critical` run | all have `reposition_to_safety` |

In [10]:
COMM_DELAY_S = 780
SEP  = '=' * 68
SEP2 = '-' * 68

HA_ICON = {
    'hold_in_place':        '[HOLD IN PLACE      ]',
    'reposition_to_safety': '[REPOSITION TO SAFETY]',
}


def _check_ha(n, label, threat, sensors, expected):
    result = choose_holding_action(threat, sensors, COMM_DELAY_S)
    status = 'PASS' if result == expected else f'FAIL (got {result!r})'
    icon   = HA_ICON[result]
    print(f'  Test {n}  {icon}  [{status}]')
    print(f'          {label}')


print(SEP)
print('  SENTINEL PROTOCOL — YELLOW Holding Action Tests')
print(SEP)

# Direct choose_holding_action() tests
_check_ha(1, 'battery_critical  charge=12%  --> reposition_to_safety',
          'battery_critical', {'charge_pct': 12.0}, 'reposition_to_safety')

_check_ha(2, 'battery_critical  charge=3%   --> hold_in_place',
          'battery_critical', {'charge_pct': 3.0},  'hold_in_place')

_check_ha(3, 'dust_storm        wind=10m/s  --> reposition_to_safety',
          'dust_storm', {'wind_speed_ms': 10.0, 'dust_density_gcm3': 0.01}, 'reposition_to_safety')

_check_ha(4, 'dust_storm        wind=22m/s  --> hold_in_place',
          'dust_storm', {'wind_speed_ms': 22.0, 'dust_density_gcm3': 0.05}, 'hold_in_place')

_check_ha(5, 'cliff_edge        any         --> hold_in_place',
          'cliff_edge', {'distance_m': 50.0, 'drift_speed_ms': 0.03},       'hold_in_place')

_check_ha(6, 'rockfall          any         --> hold_in_place',
          'rockfall', {'seismic_g': 0.3, 'debris_dist_m': 40.0, 'debris_speed_ms': 3.0},
          'hold_in_place')

_check_ha(7, 'comms_blackout    any         --> hold_in_place',
          'comms_blackout', {'relay_elevation_deg': 15.0},                   'hold_in_place')

print(SEP)

# ---------------------------------------------------------------------------
# Test 8 — full simulator run: verify every YELLOW tick carries the right action
# ---------------------------------------------------------------------------
print('  Test 8 — battery_critical full scenario run (YELLOW ticks):')
print(SEP2)
print(f'  {"Tick":>4}  {"Charge%":>8}  {"Draw/tick":>10}  {"TTH(s)":>9}  Tier          Holding Action')
print(f'  {"-"*4}  {"-"*8}  {"-"*10}  {"-"*9}  {"-"*13} {"-"*22}')

test8_pass = True
for state in run_scenario('battery_critical', ticks=20, comm_delay_s=COMM_DELAY_S):
    chg  = state.sensors['charge_pct']
    draw = state.sensors['draw_pct_per_tick']
    tth  = state.time_to_harm_s
    tier = state.tier
    ha   = state.holding_action

    tier_label = f'[{tier.value:<6}]'
    ha_label   = f'[{ha}]' if ha else '—'

    # Validate: YELLOW ticks must have a non-None holding action
    if tier == DecisionTier.YELLOW and ha is None:
        test8_pass = False
        ha_label  += '  <-- ERROR: None on YELLOW'
    # GREEN and RED must have None
    if tier != DecisionTier.YELLOW and ha is not None:
        test8_pass = False
        ha_label  += '  <-- ERROR: non-None on non-YELLOW'

    print(f'  {state.tick:>4}  {chg:>8.2f}  {draw:>10.3f}  {tth:>9.1f}  {tier_label}  {ha_label}')

print(SEP2)
print(f'  Test 8: {"PASS" if test8_pass else "FAIL"}')
print()

# Final summary
print(SEP)
print('  All holding-action tests complete.')
print(SEP)

  SENTINEL PROTOCOL — YELLOW Holding Action Tests
  Test 1  [REPOSITION TO SAFETY]  [PASS]
          battery_critical  charge=12%  --> reposition_to_safety
  Test 2  [HOLD IN PLACE      ]  [PASS]
          battery_critical  charge=3%   --> hold_in_place
  Test 3  [REPOSITION TO SAFETY]  [PASS]
          dust_storm        wind=10m/s  --> reposition_to_safety
  Test 4  [HOLD IN PLACE      ]  [PASS]
          dust_storm        wind=22m/s  --> hold_in_place
  Test 5  [HOLD IN PLACE      ]  [PASS]
          cliff_edge        any         --> hold_in_place
  Test 6  [HOLD IN PLACE      ]  [PASS]
          rockfall          any         --> hold_in_place
  Test 7  [HOLD IN PLACE      ]  [PASS]
          comms_blackout    any         --> hold_in_place
  Test 8 — battery_critical full scenario run (YELLOW ticks):
--------------------------------------------------------------------
  Tick   Charge%   Draw/tick     TTH(s)  Tier          Holding Action
  ----  --------  ----------  ---------  ------